# Weather & Time Contextual Recommendation

### This notebook looks at the weather and time components to determine suitabilityfor our egocentric activity recommendation system 

### Weather Context

The weather component will look at aspects such as temperature, precipitation, and other weather conditions to see how they affect the suitably of different activities.

### Time Context

The time component will look at the time of day it is and see how suitable an activity is based on the activity's operating hours.

## Load Dataset

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Project directory
project = Path(
    "/Users/ogechikaesedebe/Downloads/Activity-Recommendation-System"
)

# Existing activity dataset
activity_path = (
    project / "data" / "processed" / "activity_dataset.csv"
)

# Load dataset
activity_df = pd.read_csv(
    activity_path,
    low_memory=False
)

print("Dataset shape:", activity_df.shape)

print("\nAvailable columns:")
print(activity_df.columns.tolist())

print("\nFirst five rows:")
display(activity_df.head())

Dataset shape: (86577, 17)

Available columns:
['name', 'address', 'gmap_id', 'description', 'latitude', 'longitude', 'category', 'avg_rating', 'num_of_reviews', 'price', 'hours', 'MISC', 'state', 'relative_results', 'url', 'is_activity', 'activity_groups']

First five rows:


,name,address,gmap_id,description,latitude,longitude,category,avg_rating,num_of_reviews,price,hours,MISC,state,relative_results,url,is_activity,activity_groups
0,Chef Chucks Table,"Chef Chucks Table, 6128 NY-89, Ovid, NY 14521",0x89d0be3bcc338707:0x3e3d9bf1247ba191,NaN,42.727924,-76.765099,['Diner'],4.7,3,NaN,"[['Thursday', 'Closed'], ['Friday', 'Closed'],...","{'Service options': ['Delivery'], 'Amenities':...",Permanently closed,"['0x89d0eba9d9c0f817:0x1f127a3049db854d', '0x8...",https://www.google.com/maps/place//data=!4m2!3...,True,['Food & Drink']
1,Arcana Juventa Anti-Aging Spa,"Arcana Juventa Anti-Aging Spa, 14 Brighton 11t...",0x89c244688b7609cd:0x205e2d8d173dd35e,NaN,40.582130,-73.956333,"['Spa', 'Waterproofing company']",4.1,15,NaN,"[['Thursday', '10AM–8PM'], ['Friday', '10AM–8P...",NaN,Permanently closed,"['0x89c2446bdf00743d:0x1a016c5e3c603c88', '0x8...",https://www.google.com/maps/place//data=!4m2!3...,True,['Sports & Recreation']
2,Honeoye Creek Wildlife Management Area,"Honeoye Creek Wildlife Management Area, 9033 A...",0x89d13fd0bc4468e1:0x1a81c620f5086b11,NaN,42.806571,-77.525743,"['Nature preserve', 'Park']",4.7,7,NaN,NaN,NaN,NaN,"['0x89d113b7ef15c121:0xe71429a0a567a625', '0x8...",https://www.google.com/maps/place//data=!4m2!3...,True,['Outdoor & Nature']
3,VeganPro Sweets NYC,NaN,0x89c25923cbb1c20d:0xd9edcb0a146ba104,NaN,40.705649,-73.977940,"['Dessert shop', 'Service establishment']",5.0,13,NaN,"[['Thursday', 'Open 24 hours'], ['Friday', 'Op...",{'From the business': ['Identifies as women-le...,Open 24 hours,NaN,https://www.google.com/maps/place//data=!4m2!3...,True,['Food & Drink']
4,Dunkin',"Dunkin', 4008 Bell Blvd, Queens, NY 11361",0x89c261f60bdf13db:0x38da730e4687a97b,Long-running chain serving signature breakfast...,40.763985,-73.771430,"['Coffee shop', 'Bagel shop', 'Bakery', 'Break...",3.5,8,$,"[['Thursday', '6AM–7PM'], ['Friday', '6AM–7PM'...","{'Service options': ['Delivery', 'Takeout', 'D...",Open ⋅ Closes 7PM,"['0x89c3ab9229879ec3:0x3f4b2b46d7d2c503', '0x8...",https://www.google.com/maps/place//data=!4m2!3...,True,['Food & Drink']


In [2]:
import sys

print("Python executable:", sys.executable)
print("Python version:", sys.version)

Python executable: /Library/Frameworks/Python.framework/Versions/3.12/bin/python3.12
Python version: 3.12.1 (v3.12.1:2305ca5144, Dec  7 2023, 17:23:39) [Clang 13.0.0 (clang-1300.0.29.30)]


## Analyzing the available contextual metadata

In [4]:
# Columns needed for weather and time suitability
context_columns = [
    "name",
    "category",
    "activity_groups",
    "latitude",
    "longitude",
    "hours",
    "is_activity"
]

# Check missing values
metadata_summary = pd.DataFrame({
    "Non-Missing": activity_df[context_columns].notna().sum(),
    "Missing": activity_df[context_columns].isna().sum(),
    "Missing (%)": (
        activity_df[context_columns].isna().mean() * 100
    ).round(2)
})

print("CONTEXTUAL METADATA COMPLETENESS")
display(metadata_summary)

# Examine activity group distribution
print("\nACTIVITY GROUP DISTRIBUTION")
display(
    activity_df["activity_groups"]
    .value_counts(dropna=False)
    .head(20)
)

# Inspect sample category and hours values
print("\nSAMPLE ACTIVITY METADATA")
display(
    activity_df[
        ["name", "category", "activity_groups", "hours"]
    ].head(10)
)


CONTEXTUAL METADATA COMPLETENESS


,Non-Missing,Missing,Missing (%)
name,86577,0,0.00
category,86577,0,0.00
activity_groups,86577,0,0.00
latitude,86577,0,0.00
longitude,86577,0,0.00
hours,74039,12538,14.48
is_activity,86577,0,0.00



ACTIVITY GROUP DISTRIBUTION


activity_groups
['Food & Drink']                                                                                  65823
['Sports & Recreation']                                                                            4551
['Outdoor & Nature']                                                                               4464
['Arts & Culture']                                                                                 2580
['Outdoor & Nature', 'Sightseeing & Attractions']                                                  2327
['Food & Drink', 'Nightlife & Live Entertainment']                                                 1348
['Arts & Culture', 'Sightseeing & Attractions']                                                    1252
['Sightseeing & Attractions']                                                                      1162
['Games & Activities']                                                                              847
['Nightlife & Live Entertainment']              


SAMPLE ACTIVITY METADATA


,name,category,activity_groups,hours
0,Chef Chucks Table,['Diner'],['Food & Drink'],"[['Thursday', 'Closed'], ['Friday', 'Closed'],..."
1,Arcana Juventa Anti-Aging Spa,"['Spa', 'Waterproofing company']",['Sports & Recreation'],"[['Thursday', '10AM–8PM'], ['Friday', '10AM–8P..."
2,Honeoye Creek Wildlife Management Area,"['Nature preserve', 'Park']",['Outdoor & Nature'],NaN
3,VeganPro Sweets NYC,"['Dessert shop', 'Service establishment']",['Food & Drink'],"[['Thursday', 'Open 24 hours'], ['Friday', 'Op..."
4,Dunkin',"['Coffee shop', 'Bagel shop', 'Bakery', 'Break...",['Food & Drink'],"[['Thursday', '6AM–7PM'], ['Friday', '6AM–7PM'..."
5,Driscoll RD Kayak Access point,['Nature preserve'],['Outdoor & Nature'],NaN
6,Viele Pond,['Lake'],['Outdoor & Nature'],NaN
7,Tivoli Bays Management Area Parking,['Park'],['Outdoor & Nature'],NaN
8,Raffaello Kosher Pizza,['Restaurant'],['Food & Drink'],NaN
9,Casa Malinche,['Mexican restaurant'],['Food & Drink'],"[['Thursday', '11AM–9:30PM'], ['Friday', '11AM..."


## Preparing Activity Groups for Weather Context

In [5]:
import ast
from collections import Counter

# Convert string representations of lists into Python lists
def parse_activity_groups(value):
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []

activity_df["parsed_activity_groups"] = (
    activity_df["activity_groups"].apply(parse_activity_groups)
)

# Count each individual activity group
group_counts = Counter(
    group
    for groups in activity_df["parsed_activity_groups"]
    for group in groups
)

group_summary = pd.DataFrame(
    group_counts.items(),
    columns=["Activity Group", "Number of Activities"]
).sort_values(
    "Number of Activities",
    ascending=False
).reset_index(drop=True)

print("INDIVIDUAL ACTIVITY GROUP DISTRIBUTION")
display(group_summary)

print("\nSAMPLE PARSED ACTIVITY GROUPS")
display(
    activity_df[
        ["name", "activity_groups", "parsed_activity_groups"]
    ].head(10)
)

print(
    "\nActivities with no parsed groups:",
    (activity_df["parsed_activity_groups"].str.len() == 0).sum()
)

INDIVIDUAL ACTIVITY GROUP DISTRIBUTION


,Activity Group,Number of Activities
0,Food & Drink,67820
1,Outdoor & Nature,7423
2,Sightseeing & Attractions,5478
3,Sports & Recreation,5196
4,Arts & Culture,4120
5,Nightlife & Live Entertainment,2086
6,Games & Activities,1650



SAMPLE PARSED ACTIVITY GROUPS


,name,activity_groups,parsed_activity_groups
0,Chef Chucks Table,['Food & Drink'],[Food & Drink]
1,Arcana Juventa Anti-Aging Spa,['Sports & Recreation'],[Sports & Recreation]
2,Honeoye Creek Wildlife Management Area,['Outdoor & Nature'],[Outdoor & Nature]
3,VeganPro Sweets NYC,['Food & Drink'],[Food & Drink]
4,Dunkin',['Food & Drink'],[Food & Drink]
5,Driscoll RD Kayak Access point,['Outdoor & Nature'],[Outdoor & Nature]
6,Viele Pond,['Outdoor & Nature'],[Outdoor & Nature]
7,Tivoli Bays Management Area Parking,['Outdoor & Nature'],[Outdoor & Nature]
8,Raffaello Kosher Pizza,['Food & Drink'],[Food & Drink]
9,Casa Malinche,['Food & Drink'],[Food & Drink]



Activities with no parsed groups: 0


## Examining Detailed Activity Categories

In [6]:
# Reuse the list-parsing function from Step 3
activity_df["parsed_categories"] = (
    activity_df["category"].apply(parse_activity_groups)
)

# Count individual detailed categories
category_counts = Counter(
    category
    for categories in activity_df["parsed_categories"]
    for category in categories
)

category_summary = pd.DataFrame(
    category_counts.items(),
    columns=["Category", "Number of Activities"]
).sort_values(
    "Number of Activities",
    ascending=False
).reset_index(drop=True)

print("TOTAL UNIQUE DETAILED CATEGORIES:")
print(len(category_summary))

print("\nTOP 40 DETAILED CATEGORIES:")
display(category_summary.head(40))

print("\nSAMPLE ACTIVITIES WITH DETAILED CATEGORIES:")
display(
    activity_df[
        ["name", "parsed_categories", "parsed_activity_groups"]
    ].sample(15, random_state=42)
)

TOTAL UNIQUE DETAILED CATEGORIES:
1604

TOP 40 DETAILED CATEGORIES:


,Category,Number of Activities
0,Restaurant,18828
1,Bar,7704
2,Fast food restaurant,6437
3,Pizza restaurant,5996
4,Delivery Restaurant,5754
5,Coffee shop,5582
6,Takeout Restaurant,5370
7,American restaurant,5163
8,Bakery,4854
9,Tourist attraction,4715



SAMPLE ACTIVITIES WITH DETAILED CATEGORIES:


,name,parsed_categories,parsed_activity_groups
67915,Atlantic Diner,[Diner],[Food & Drink]
62777,Helium Comedy Club,"[Comedy club, Bar, Night club, Performing arts...","[Food & Drink, Arts & Culture, Nightlife & Liv..."
48973,Oslo Coffee Roasters,"[Coffee shop, Cafe, Espresso bar]",[Food & Drink]
81010,Popeyes Louisiana Kitchen,"[Chicken restaurant, Cajun restaurant, Fast fo...",[Food & Drink]
58893,Mumbai Place,[Indian restaurant],[Food & Drink]
52040,Monks on the Commons,"[American restaurant, Bar]",[Food & Drink]
80398,Mokum,[Restaurant],[Food & Drink]
35841,Cheri's Bedstuy,[Southern restaurant (US)],[Food & Drink]
66161,Good Eats,"[Diner, American restaurant, Hamburger restaur...",[Food & Drink]
6748,Pizza Plus,"[Pizza restaurant, Restaurant]",[Food & Drink]


## Initial Weather Exposure Classification

In [7]:
# Initial category keywords indicating outdoor activities
outdoor_keywords = [
    "park",
    "hiking",
    "trail",
    "nature preserve",
    "beach",
    "botanical garden",
    "campground",
    "outdoor swimming",
    "kayak",
    "marina",
    "golf course",
    "playground"
]

# Initial category keywords indicating indoor activities
indoor_keywords = [
    "museum",
    "art gallery",
    "spa",
    "movie theater",
    "cinema",
    "bowling alley",
    "indoor playground",
    "fitness center",
    "gym",
    "library",
    "aquarium",
    "escape room",
    "comedy club"
]

def classify_weather_exposure(categories):
    category_text = [
        str(category).lower()
        for category in categories
    ]

    outdoor_match = any(
        keyword in category
        for category in category_text
        for keyword in outdoor_keywords
    )

    indoor_match = any(
        keyword in category
        for category in category_text
        for keyword in indoor_keywords
    )

    if outdoor_match and not indoor_match:
        return "outdoor"

    if indoor_match and not outdoor_match:
        return "indoor"

    return "unknown"


activity_df["weather_exposure"] = (
    activity_df["parsed_categories"]
    .apply(classify_weather_exposure)
)

print("INITIAL WEATHER EXPOSURE DISTRIBUTION")
display(
    activity_df["weather_exposure"]
    .value_counts()
    .to_frame("Number of Activities")
)

print("\nWEATHER EXPOSURE PERCENTAGES")
display(
    activity_df["weather_exposure"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .to_frame("Percentage (%)")
)

print("\nSAMPLE CLASSIFIED ACTIVITIES")
display(
    activity_df[
        ["name", "parsed_categories", "weather_exposure"]
    ].sample(20, random_state=42)
)

INITIAL WEATHER EXPOSURE DISTRIBUTION


,Number of Activities
weather_exposure,
unknown,72775
outdoor,8185
indoor,5617



WEATHER EXPOSURE PERCENTAGES


,Percentage (%)
weather_exposure,
unknown,84.06
outdoor,9.45
indoor,6.49



SAMPLE CLASSIFIED ACTIVITIES


,name,parsed_categories,weather_exposure
67915,Atlantic Diner,[Diner],unknown
62777,Helium Comedy Club,"[Comedy club, Bar, Night club, Performing arts...",indoor
48973,Oslo Coffee Roasters,"[Coffee shop, Cafe, Espresso bar]",unknown
81010,Popeyes Louisiana Kitchen,"[Chicken restaurant, Cajun restaurant, Fast fo...",unknown
58893,Mumbai Place,[Indian restaurant],unknown
52040,Monks on the Commons,"[American restaurant, Bar]",unknown
80398,Mokum,[Restaurant],unknown
35841,Cheri's Bedstuy,[Southern restaurant (US)],unknown
66161,Good Eats,"[Diner, American restaurant, Hamburger restaur...",unknown
6748,Pizza Plus,"[Pizza restaurant, Restaurant]",unknown


## Validating Weather Exposure Classification

In [8]:
# Inspect a reproducible sample of outdoor activities
outdoor_sample = (
    activity_df[
        activity_df["weather_exposure"] == "outdoor"
    ][["name", "parsed_categories", "parsed_activity_groups"]]
    .sample(15, random_state=42)
)

print("SAMPLE OUTDOOR ACTIVITIES")
display(outdoor_sample)

# Inspect a reproducible sample of indoor activities
indoor_sample = (
    activity_df[
        activity_df["weather_exposure"] == "indoor"
    ][["name", "parsed_categories", "parsed_activity_groups"]]
    .sample(15, random_state=42)
)

print("\nSAMPLE INDOOR ACTIVITIES")
display(indoor_sample)

SAMPLE OUTDOOR ACTIVITIES


,name,parsed_categories,parsed_activity_groups
46030,Horace Harding Playground,[Playground],[Games & Activities]
55737,Kennedy Memorial Park,"[Park, Tourist attraction]","[Outdoor & Nature, Sightseeing & Attractions]"
25999,Calvin Engle Park,[Park],[Outdoor & Nature]
26388,Stehli Town Beach,[Beach],[Outdoor & Nature]
11140,Greens At Beaumont,[Golf course],[Sports & Recreation]
38715,Triborough Bridge Playground A,[Park],[Outdoor & Nature]
25012,Spaulding Lake Club,[Park],[Outdoor & Nature]
70073,Ashokan Rail Trail- Ashokan Station Trailhead,[Hiking area],[Sports & Recreation]
60668,Cooney Grauer Field,"[Park, Baseball field, Tourist attraction]","[Outdoor & Nature, Sports & Recreation, Sights..."
29335,MacDonough Park,[Park],[Outdoor & Nature]



SAMPLE INDOOR ACTIVITIES


,name,parsed_categories,parsed_activity_groups
49893,Sangarita's Tapas Wine,"[Spanish restaurant, Bar, Lounge, Tapas bar, T...","[Food & Drink, Nightlife & Live Entertainment]"
32989,PRESS Massage Williamsburg,"[Massage spa, Massage therapist, Wellness center]",[Sports & Recreation]
611,Red Sun Foot Spa L&C,[Massage spa],[Sports & Recreation]
17938,Parking lot,[Museum],[Arts & Culture]
82823,Museum of American Armor,"[Museum, Tourist attraction]","[Arts & Culture, Sightseeing & Attractions]"
2661,Nunda Historical Society,"[Local history museum, Art gallery, Tourist at...","[Arts & Culture, Sightseeing & Attractions]"
53751,New Attitude Salon and Wellness Spa,"[Massage therapist, Facial spa, Hair salon, Ma...",[Sports & Recreation]
17121,Brennan & Griffin,[Art gallery],[Arts & Culture]
15438,Newman Plastic Surgery & Laser Center: Scott N...,"[Plastic surgeon, Laser hair removal service, ...",[Sports & Recreation]
6932,Jose Luis,"[Italian restaurant, Mediterranean restaurant,...",[Food & Drink]


### Notes:

##### Weather exposure classification: Our keyword matching is too broad. For example, "park" matches "business park".



##### Activity relevance: Some records may not be appropriate recreational recommendations, even when their indoor/outdoor classification is technically correct.

## Refining Weather Exposure Classification

In [9]:
# Specific outdoor category labels
outdoor_categories = {
    "park",
    "playground",
    "beach",
    "nature preserve",
    "hiking area",
    "campground",
    "golf course",
    "botanical garden",
    "marina",
    "kayak rental service",
    "baseball field",
    "basketball court",
    "soccer field",
    "tennis court",
    "state park",
    "national park",
    "wildlife refuge"
}

# Specific indoor category labels
indoor_categories = {
    "museum",
    "art museum",
    "history museum",
    "art gallery",
    "spa",
    "day spa",
    "massage spa",
    "movie theater",
    "cinema",
    "bowling alley",
    "gym",
    "fitness center",
    "library",
    "aquarium",
    "escape room center",
    "comedy club"
}

def classify_weather_exposure_v2(categories):
    normalized = {
        str(category).strip().lower()
        for category in categories
    }

    outdoor_match = bool(normalized & outdoor_categories)
    indoor_match = bool(normalized & indoor_categories)

    if outdoor_match and not indoor_match:
        return "outdoor"

    if indoor_match and not outdoor_match:
        return "indoor"

    return "unknown"


activity_df["weather_exposure_v2"] = (
    activity_df["parsed_categories"]
    .apply(classify_weather_exposure_v2)
)

print("REVISED WEATHER EXPOSURE DISTRIBUTION")
display(
    activity_df["weather_exposure_v2"]
    .value_counts()
    .to_frame("Number of Activities")
)

print("\nCLASSIFICATION CHANGES")
display(
    pd.crosstab(
        activity_df["weather_exposure"],
        activity_df["weather_exposure_v2"],
        rownames=["Original"],
        colnames=["Revised"]
    )
)

print("\nSAMPLE REVISED CLASSIFICATIONS")
display(
    activity_df[
        ["name", "parsed_categories", "weather_exposure_v2"]
    ].sample(20, random_state=42)
)

REVISED WEATHER EXPOSURE DISTRIBUTION


,Number of Activities
weather_exposure_v2,
unknown,74484
outdoor,7620
indoor,4473



CLASSIFICATION CHANGES


Revised,indoor,outdoor,unknown
Original,,,
indoor,4456,1,1160
outdoor,0,7161,1024
unknown,17,458,72300



SAMPLE REVISED CLASSIFICATIONS


,name,parsed_categories,weather_exposure_v2
67915,Atlantic Diner,[Diner],unknown
62777,Helium Comedy Club,"[Comedy club, Bar, Night club, Performing arts...",indoor
48973,Oslo Coffee Roasters,"[Coffee shop, Cafe, Espresso bar]",unknown
81010,Popeyes Louisiana Kitchen,"[Chicken restaurant, Cajun restaurant, Fast fo...",unknown
58893,Mumbai Place,[Indian restaurant],unknown
52040,Monks on the Commons,"[American restaurant, Bar]",unknown
80398,Mokum,[Restaurant],unknown
35841,Cheri's Bedstuy,[Southern restaurant (US)],unknown
66161,Good Eats,"[Diner, American restaurant, Hamburger restaur...",unknown
6748,Pizza Plus,"[Pizza restaurant, Restaurant]",unknown


## Initial Weather Suitability Scoring

In [11]:
# Initial heuristic weather suitability scores
weather_score_rules = {
    "clear": {
        "outdoor": 1.0,
        "indoor": 0.9,
        "unknown": 0.7
    },
    "rain": {
        "outdoor": 0.2,
        "indoor": 1.0,
        "unknown": 0.7
    }
}

def calculate_weather_score(exposure, weather_condition):
    if weather_condition not in weather_score_rules:
        raise ValueError(
            f"Unsupported weather condition: {weather_condition}"
        )

    return weather_score_rules[weather_condition].get(
        exposure, 0.7
    )


# Calculate scores for two weather scenarios
activity_df["weather_score_clear"] = (
    activity_df["weather_exposure_v2"]
    .apply(lambda x: calculate_weather_score(x, "clear"))
)

activity_df["weather_score_rain"] = (
    activity_df["weather_exposure_v2"]
    .apply(lambda x: calculate_weather_score(x, "rain"))
)

print("WEATHER SUITABILITY SCORES BY EXPOSURE")
display(
    activity_df.groupby("weather_exposure_v2")[
        ["weather_score_clear", "weather_score_rain"]
    ].agg(["count", "mean"])
)

print("\nSAMPLE ACTIVITY WEATHER SCORES")
display(
    activity_df[
        [
            "name",
            "weather_exposure_v2",
            "weather_score_clear",
            "weather_score_rain"
        ]
    ].sample(20, random_state=42)
)

WEATHER SUITABILITY SCORES BY EXPOSURE


weather_score_clear      weather_score_rain     
                                  count mean              count mean
weather_exposure_v2                                                 
indoor                             4473  0.9               4473  1.0
outdoor                            7620  1.0               7620  0.2
unknown                           74484  0.7              74484  0.7


SAMPLE ACTIVITY WEATHER SCORES


,name,weather_exposure_v2,weather_score_clear,weather_score_rain
67915,Atlantic Diner,unknown,0.7,0.7
62777,Helium Comedy Club,indoor,0.9,1.0
48973,Oslo Coffee Roasters,unknown,0.7,0.7
81010,Popeyes Louisiana Kitchen,unknown,0.7,0.7
58893,Mumbai Place,unknown,0.7,0.7
52040,Monks on the Commons,unknown,0.7,0.7
80398,Mokum,unknown,0.7,0.7
35841,Cheri's Bedstuy,unknown,0.7,0.7
66161,Good Eats,unknown,0.7,0.7
6748,Pizza Plus,unknown,0.7,0.7


## Incorporating Temperature into Weather Suitability

In [13]:
def calculate_temperature_factor(temperature_c):
    if temperature_c < 0:
        return 0.2
    elif temperature_c < 10:
        return 0.5
    elif temperature_c < 30:
        return 1.0
    elif temperature_c < 35:
        return 0.6
    else:
        return 0.2


def calculate_combined_weather_score(
    exposure,
    weather_condition,
    temperature_c
):
    precipitation_score = calculate_weather_score(
        exposure,
        weather_condition
    )

    if exposure == "outdoor":
        temperature_factor = calculate_temperature_factor(
            temperature_c
        )

        return round(
            precipitation_score * temperature_factor,
            3
        )

    return precipitation_score


# Test the model using different weather scenarios
weather_scenarios = [
    ("Clear, comfortable", "clear", 22),
    ("Clear, freezing", "clear", -5),
    ("Clear, very hot", "clear", 38),
    ("Rainy, comfortable", "rain", 20),
    ("Rainy, cold", "rain", 5)
]

results = []

for scenario_name, condition, temperature in weather_scenarios:
    for exposure in ["outdoor", "indoor", "unknown"]:
        score = calculate_combined_weather_score(
            exposure,
            condition,
            temperature
        )

        results.append({
            "Scenario": scenario_name,
            "Exposure": exposure,
            "Temperature (°C)": temperature,
            "Weather Score": score
        })

weather_scenario_df = pd.DataFrame(results)

print("WEATHER SUITABILITY ACROSS SCENARIOS")
display(
    weather_scenario_df.pivot(
        index="Scenario",
        columns="Exposure",
        values="Weather Score"
    )
)

WEATHER SUITABILITY ACROSS SCENARIOS


Exposure,indoor,outdoor,unknown
Scenario,,,
"Clear, comfortable",0.9,1.0,0.7
"Clear, freezing",0.9,0.2,0.7
"Clear, very hot",0.9,0.2,0.7
"Rainy, cold",1.0,0.1,0.7
"Rainy, comfortable",1.0,0.2,0.7


## Examining Operating Hours for Time Context

In [14]:
# Select activities with available operating hours
hours_available = activity_df[
    activity_df["hours"].notna()
]

print("ACTIVITIES WITH OPERATING HOURS:")
print(len(hours_available))

print("\nSAMPLE OPERATING HOURS:")
for _, row in hours_available[
    ["name", "hours"]
].sample(10, random_state=42).iterrows():
    print(f"\nActivity: {row['name']}")
    print(f"Hours: {row['hours']}")

print("\nACTIVITIES WITH MISSING OPERATING HOURS:")
display(
    activity_df[
        activity_df["hours"].isna()
    ][
        ["name", "parsed_categories", "parsed_activity_groups"]
    ].head(10)
)

ACTIVITIES WITH OPERATING HOURS:
74039

SAMPLE OPERATING HOURS:

Activity: Corner Deli
Hours: [['Saturday', '6AM–4PM'], ['Sunday', '7AM–4PM'], ['Monday', '6AM–5PM'], ['Tuesday', '6AM–5PM'], ['Wednesday', '6AM–5PM'], ['Thursday', '6AM–5PM'], ['Friday', '6AM–5PM']]

Activity: Veterans Memorial Park
Hours: [['Tuesday', 'Open 24 hours'], ['Wednesday', 'Open 24 hours'], ['Thursday', 'Open 24 hours'], ['Friday', 'Open 24 hours'], ['Saturday', 'Open 24 hours'], ['Sunday', 'Open 24 hours'], ['Monday', 'Open 24 hours']]

Activity: Mini Market Best Way Deli
Hours: [['Thursday', 'Open 24 hours'], ['Friday', 'Open 24 hours'], ['Saturday', 'Open 24 hours'], ['Sunday', '8AM–12AM'], ['Monday', 'Open 24 hours'], ['Tuesday', 'Open 24 hours'], ['Wednesday', 'Open 24 hours']]

Activity: Antique Carousel
Hours: [['Friday', '4–9PM'], ['Saturday', '10AM–6PM'], ['Sunday', '10AM–6PM'], ['Monday', 'Closed'], ['Tuesday', 'Closed'], ['Wednesday', 'Closed'], ['Thursday', 'Closed']]

Activity: Food King
Hours: [['

,name,parsed_categories,parsed_activity_groups
2,Honeoye Creek Wildlife Management Area,"[Nature preserve, Park]",[Outdoor & Nature]
5,Driscoll RD Kayak Access point,[Nature preserve],[Outdoor & Nature]
6,Viele Pond,[Lake],[Outdoor & Nature]
7,Tivoli Bays Management Area Parking,[Park],[Outdoor & Nature]
8,Raffaello Kosher Pizza,[Restaurant],[Food & Drink]
12,Claudio's Lounge,[Cocktail bar],[Food & Drink]
14,Cafe Kristall,"[Cafe, Restaurant]",[Food & Drink]
15,Liberty Theater,[Movie theater],[Arts & Culture]
19,Fairmount Playground,"[Park, Tourist attraction]","[Outdoor & Nature, Sightseeing & Attractions]"
20,Lackawanna Steel Company,[Historical landmark],"[Arts & Culture, Sightseeing & Attractions]"


## Parsing Weekly Operating Hours

In [15]:
import ast

WEEKDAYS = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

def parse_weekly_hours(value):
    if isinstance(value, str):
        try:
            value = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return {}

    if not isinstance(value, list):
        return {}

    weekly_hours = {}

    for entry in value:
        if (
            isinstance(entry, (list, tuple))
            and len(entry) == 2
        ):
            day, hours = entry

            if (
                isinstance(day, str)
                and day in WEEKDAYS
                and isinstance(hours, str)
            ):
                weekly_hours[day] = hours.strip()

    return weekly_hours


activity_df["parsed_hours"] = (
    activity_df["hours"].apply(parse_weekly_hours)
)

print("OPERATING HOURS PARSING SUMMARY")

print(
    "Activities with parsed schedules:",
    (activity_df["parsed_hours"].apply(len) > 0).sum()
)

print(
    "Activities without parsed schedules:",
    (activity_df["parsed_hours"].apply(len) == 0).sum()
)

print("\nSAMPLE PARSED SCHEDULES")

display(
    activity_df[
        ["name", "parsed_hours"]
    ].sample(10, random_state=42)
)

print("\nEXAMPLE: ANTIQUE CAROUSEL")

display(
    activity_df[
        activity_df["name"] == "Antique Carousel"
    ][["name", "parsed_hours"]].head()
)

OPERATING HOURS PARSING SUMMARY
Activities with parsed schedules: 74039
Activities without parsed schedules: 12538

SAMPLE PARSED SCHEDULES


,name,parsed_hours
67915,Atlantic Diner,"{'Sunday': 'Open 24 hours', 'Monday': 'Open 24..."
62777,Helium Comedy Club,"{'Thursday': 'Closed', 'Friday': '5–11:30PM', ..."
48973,Oslo Coffee Roasters,"{'Sunday': '8AM–3PM', 'Monday': '8AM–3PM', 'Tu..."
81010,Popeyes Louisiana Kitchen,"{'Sunday': '10:30AM–8PM', 'Monday': '10:30AM–8..."
58893,Mumbai Place,"{'Monday': '12–10:15PM', 'Tuesday': '12–10:15P..."
52040,Monks on the Commons,"{'Friday': '3–11PM', 'Saturday': '9AM–11PM', '..."
80398,Mokum,{}
35841,Cheri's Bedstuy,"{'Tuesday': 'Closed', 'Wednesday': '11:30AM–10..."
66161,Good Eats,"{'Monday': '8AM–8:30PM', 'Tuesday': '8AM–8:30P..."
6748,Pizza Plus,"{'Monday': '4–6PM', 'Tuesday': '11AM–8PM', 'We..."



EXAMPLE: ANTIQUE CAROUSEL


,name,parsed_hours
76604,Antique Carousel,"{'Friday': '4–9PM', 'Saturday': '10AM–6PM', 'S..."


##  Converting Operating Hours into Numerical Time Intervals


In [16]:
import re

def parse_clock_time(time_text, default_meridiem=None):
    """Convert a 12-hour clock time to minutes after midnight."""
    match = re.fullmatch(
        r"\s*(1[0-2]|[1-9])(?::([0-5]\d))?\s*(AM|PM)?\s*",
        time_text,
        flags=re.IGNORECASE
    )

    if not match:
        return None

    hour = int(match.group(1))
    minute = int(match.group(2) or 0)
    meridiem = (match.group(3) or default_meridiem)

    if meridiem is None:
        return None

    meridiem = meridiem.upper()

    hour = hour % 12
    if meridiem == "PM":
        hour += 12

    return hour * 60 + minute


def parse_time_range(hours_text):
    """Return (opening_minute, closing_minute) or a status."""
    text = hours_text.strip()

    if text.lower() == "closed":
        return "closed"

    if text.lower() == "open 24 hours":
        return (0, 1440)

    parts = re.split(r"\s*[–—-]\s*", text)

    if len(parts) != 2:
        return "unknown"

    start_text, end_text = parts

    end_match = re.search(
        r"(AM|PM)\s*$", end_text, flags=re.IGNORECASE
    )

    if not end_match:
        return "unknown"

    end_meridiem = end_match.group(1).upper()

    # If the opening time has no AM/PM, infer it from
    # the closing time, then adjust if necessary.
    start_has_meridiem = bool(
        re.search(r"(AM|PM)\s*$", start_text, flags=re.IGNORECASE)
    )

    end_minute = parse_clock_time(end_text)

    if start_has_meridiem:
        start_minute = parse_clock_time(start_text)
    else:
        start_minute = parse_clock_time(
            start_text, default_meridiem=end_meridiem
        )

        # Example: 11–2PM means 11AM–2PM.
        if (
            start_minute is not None
            and end_minute is not None
            and start_minute >= end_minute
        ):
            opposite = "AM" if end_meridiem == "PM" else "PM"
            alternative = parse_clock_time(
                start_text, default_meridiem=opposite
            )

            if alternative is not None and alternative < end_minute:
                start_minute = alternative

    if start_minute is None or end_minute is None:
        return "unknown"

    # Closing at or before opening means the interval
    # extends into the following day.
    if end_minute <= start_minute:
        end_minute += 1440

    return (start_minute, end_minute)


# Test the parser on representative examples
test_hours = [
    "10AM–6PM",
    "4–9PM",
    "11AM–9:30PM",
    "8AM–12AM",
    "11–2PM",
    "5:30–11PM",
    "12–10PM",
    "Closed",
    "Open 24 hours"
]

test_results = pd.DataFrame({
    "Original Hours": test_hours,
    "Parsed Result": [
        str(parse_time_range(value))
        for value in test_hours
    ]
})

display(test_results)

,Original Hours,Parsed Result
0,10AM–6PM,"(600, 1080)"
1,4–9PM,"(960, 1260)"
2,11AM–9:30PM,"(660, 1290)"
3,8AM–12AM,"(480, 1440)"
4,11–2PM,"(660, 840)"
5,5:30–11PM,"(1050, 1380)"
6,12–10PM,"(720, 1320)"
7,Closed,closed
8,Open 24 hours,"(0, 1440)"


## Time-Based Activity Availability

In [17]:
def check_activity_availability(weekly_hours, day, hour, minute=0):
    if day not in WEEKDAYS:
        raise ValueError(f"Invalid weekday: {day}")

    if not (0 <= hour <= 23 and 0 <= minute <= 59):
        raise ValueError("Invalid time")

    if not weekly_hours:
        return "unknown"

    requested_minute = hour * 60 + minute
    day_index = WEEKDAYS.index(day)
    previous_day = WEEKDAYS[(day_index - 1) % 7]

    # Check today's schedule
    today_hours = weekly_hours.get(day)

    if today_hours is None:
        today_status = "unknown"
    else:
        today_interval = parse_time_range(today_hours)

        if today_interval == "closed":
            today_status = "closed"
        elif today_interval == "unknown":
            today_status = "unknown"
        elif (
            today_interval[0]
            <= requested_minute
            < today_interval[1]
        ):
            return "open"
        else:
            today_status = "closed"

    # Check overnight opening hours from the previous day
    previous_hours = weekly_hours.get(previous_day)

    if previous_hours is not None:
        previous_interval = parse_time_range(previous_hours)

        if isinstance(previous_interval, tuple):
            if (
                previous_interval[1] > 1440
                and requested_minute
                < previous_interval[1] - 1440
            ):
                return "open"

        elif previous_interval == "unknown":
            if today_status != "closed":
                return "unknown"

    return today_status


# Test cases
test_schedules = [
    (
        "Standard opening",
        {"Monday": "9AM–5PM"},
        "Monday", 11, 0
    ),
    (
        "After closing",
        {"Monday": "9AM–5PM"},
        "Monday", 19, 0
    ),
    (
        "Closed day",
        {"Monday": "Closed"},
        "Monday", 12, 0
    ),
    (
        "24-hour opening",
        {"Monday": "Open 24 hours"},
        "Monday", 23, 0
    ),
    (
        "Overnight opening",
        {"Friday": "8PM–2AM", "Saturday": "Closed"},
        "Saturday", 1, 0
    ),
    (
        "Missing hours",
        {},
        "Monday", 12, 0
    )
]

test_results = []

for name, schedule, day, hour, minute in test_schedules:
    result = check_activity_availability(
        schedule, day, hour, minute
    )

    test_results.append({
        "Test Case": name,
        "Requested Day": day,
        "Requested Time": f"{hour:02d}:{minute:02d}",
        "Availability": result
    })

display(pd.DataFrame(test_results))

,Test Case,Requested Day,Requested Time,Availability
0,Standard opening,Monday,11:00,open
1,After closing,Monday,19:00,closed
2,Closed day,Monday,12:00,closed
3,24-hour opening,Monday,23:00,open
4,Overnight opening,Saturday,01:00,open
5,Missing hours,Monday,12:00,unknown


## Numerical Time Availability Scoring

In [18]:
# Convert availability status into a numerical score
time_score_rules = {
    "open": 1.0,
    "closed": 0.0,
    "unknown": 0.5
}

def calculate_time_score(weekly_hours, day, hour, minute=0):
    availability = check_activity_availability(
        weekly_hours,
        day,
        hour,
        minute
    )

    return time_score_rules[availability]


# Test using a hypothetical recommendation context
requested_day = "Saturday"
requested_hour = 14
requested_minute = 0

# Calculate availability and time scores for all activities
activity_df["time_availability"] = (
    activity_df["parsed_hours"].apply(
        lambda hours: check_activity_availability(
            hours,
            requested_day,
            requested_hour,
            requested_minute
        )
    )
)

activity_df["time_score"] = (
    activity_df["time_availability"].map(time_score_rules)
)

print("TIME AVAILABILITY DISTRIBUTION")
display(
    activity_df["time_availability"]
    .value_counts()
    .to_frame("Number of Activities")
)

print("\nTIME SCORE DISTRIBUTION")
display(
    activity_df["time_score"]
    .value_counts()
    .sort_index()
    .to_frame("Number of Activities")
)

print("\nSAMPLE ACTIVITY TIME SCORES")
display(
    activity_df[
        ["name", "time_availability", "time_score"]
    ].sample(20, random_state=42)
)

TIME AVAILABILITY DISTRIBUTION


,Number of Activities
time_availability,
open,62643
unknown,12538
closed,11396



TIME SCORE DISTRIBUTION


,Number of Activities
time_score,
0.0,11396
0.5,12538
1.0,62643



SAMPLE ACTIVITY TIME SCORES


,name,time_availability,time_score
67915,Atlantic Diner,open,1.0
62777,Helium Comedy Club,closed,0.0
48973,Oslo Coffee Roasters,open,1.0
81010,Popeyes Louisiana Kitchen,open,1.0
58893,Mumbai Place,open,1.0
52040,Monks on the Commons,open,1.0
80398,Mokum,unknown,0.5
35841,Cheri's Bedstuy,open,1.0
66161,Good Eats,open,1.0
6748,Pizza Plus,open,1.0


## Auditing Operating-Hours Parsing

In [19]:
from collections import Counter

# Collect every operating-hours description
all_hour_descriptions = [
    description
    for schedule in activity_df["parsed_hours"]
    for description in schedule.values()
]

# Count parser results
parse_status_counts = Counter()
unrecognized_hours = Counter()

for description in all_hour_descriptions:
    result = parse_time_range(description)

    if result == "unknown":
        parse_status_counts["unknown"] += 1
        unrecognized_hours[description] += 1

    elif result == "closed":
        parse_status_counts["closed"] += 1

    elif isinstance(result, tuple):
        parse_status_counts["parsed_interval"] += 1


print("OPERATING-HOURS PARSING AUDIT")
display(
    pd.DataFrame(
        parse_status_counts.items(),
        columns=["Parsing Status", "Number of Day Entries"]
    )
)

print("\nTOP 20 UNRECOGNIZED HOUR FORMATS")
display(
    pd.DataFrame(
        unrecognized_hours.most_common(20),
        columns=["Hours Description", "Frequency"]
    )
)

print("\nTOTAL UNRECOGNIZED DAY ENTRIES:")
print(sum(unrecognized_hours.values()))

print("\nUNIQUE UNRECOGNIZED FORMATS:")
print(len(unrecognized_hours))

OPERATING-HOURS PARSING AUDIT


,Parsing Status,Number of Day Entries
0,closed,39479
1,parsed_interval,478794



TOP 20 UNRECOGNIZED HOUR FORMATS


,Hours Description,Frequency



TOTAL UNRECOGNIZED DAY ENTRIES:
0

UNIQUE UNRECOGNIZED FORMATS:
0


## Combining Weather and Time Suitability

In [20]:
# Hypothetical recommendation scenario
scenario_weather = "clear"
scenario_temperature = 22

# Calculate weather suitability
activity_df["scenario_weather_score"] = (
    activity_df["weather_exposure_v2"].apply(
        lambda exposure: calculate_combined_weather_score(
            exposure,
            scenario_weather,
            scenario_temperature
        )
    )
)

# Combine weather and time scores
weather_weight = 0.5
time_weight = 0.5

activity_df["context_score"] = (
    weather_weight * activity_df["scenario_weather_score"]
    + time_weight * activity_df["time_score"]
)

# Keep confirmed closure separate from the weighted score
activity_df["context_eligible"] = (
    activity_df["time_availability"] != "closed"
)

print("CONTEXT SCORE SUMMARY")
display(activity_df["context_score"].describe())

print("\nELIGIBILITY DISTRIBUTION")
display(
    activity_df["context_eligible"]
    .value_counts()
    .to_frame("Number of Activities")
)

print("\nSAMPLE COMBINED CONTEXT SCORES")
display(
    activity_df[
        [
            "name",
            "weather_exposure_v2",
            "scenario_weather_score",
            "time_availability",
            "time_score",
            "context_score",
            "context_eligible"
        ]
    ].sample(20, random_state=42)
)

CONTEXT SCORE SUMMARY


count    86577.000000
mean         0.766350
std          0.180562
min          0.350000
25%          0.750000
50%          0.850000
75%          0.850000
max          1.000000
Name: context_score, dtype: float64


ELIGIBILITY DISTRIBUTION


,Number of Activities
context_eligible,
True,75181
False,11396



SAMPLE COMBINED CONTEXT SCORES


,name,weather_exposure_v2,scenario_weather_score,time_availability,time_score,context_score,context_eligible
67915,Atlantic Diner,unknown,0.7,open,1.0,0.85,True
62777,Helium Comedy Club,indoor,0.9,closed,0.0,0.45,False
48973,Oslo Coffee Roasters,unknown,0.7,open,1.0,0.85,True
81010,Popeyes Louisiana Kitchen,unknown,0.7,open,1.0,0.85,True
58893,Mumbai Place,unknown,0.7,open,1.0,0.85,True
52040,Monks on the Commons,unknown,0.7,open,1.0,0.85,True
80398,Mokum,unknown,0.7,unknown,0.5,0.60,True
35841,Cheri's Bedstuy,unknown,0.7,open,1.0,0.85,True
66161,Good Eats,unknown,0.7,open,1.0,0.85,True
6748,Pizza Plus,unknown,0.7,open,1.0,0.85,True


## Comparing Contextual Rankings Across Weather Scenarios

In [22]:
# Compare two weather scenarios
scenarios = {
    "clear_comfortable": ("clear", 22),
    "rainy_cold": ("rain", 5)
}

for scenario_name, (condition, temperature) in scenarios.items():

    weather_scores = activity_df["weather_exposure_v2"].apply(
        lambda exposure: calculate_combined_weather_score(
            exposure,
            condition,
            temperature
        )
    )

    activity_df[f"context_{scenario_name}"] = (
        0.5 * weather_scores
        + 0.5 * activity_df["time_score"]
    )

# Select illustrative activities from different exposure groups
example_names = [
    "Indian Head Trailhead",
    "Helium Comedy Club",
    "Oslo Coffee Roasters",
    "Kennedy Memorial Park",
    "Museum of American Armor",
    "Atlantic Diner"
]

comparison_df = activity_df[
    activity_df["name"].isin(example_names)
][
    [
        "name",
        "weather_exposure_v2",
        "time_availability",
        "context_eligible",
        "context_clear_comfortable",
        "context_rainy_cold"
    ]
].copy()

# Exclude confirmed-closed activities
comparison_df = comparison_df[
    comparison_df["context_eligible"]
]

comparison_df["score_change"] = (
    comparison_df["context_rainy_cold"]
    - comparison_df["context_clear_comfortable"]
)

print("CONTEXTUAL SUITABILITY COMPARISON")
display(
    comparison_df.sort_values(
        "context_clear_comfortable",
        ascending=False
    )
)

CONTEXTUAL SUITABILITY COMPARISON


,name,weather_exposure_v2,time_availability,context_eligible,context_clear_comfortable,context_rainy_cold,score_change
55737,Kennedy Memorial Park,outdoor,open,True,1.00,0.55,-0.45
82823,Museum of American Armor,indoor,open,True,0.95,1.00,0.05
60351,Oslo Coffee Roasters,unknown,open,True,0.85,0.85,0.00
48973,Oslo Coffee Roasters,unknown,open,True,0.85,0.85,0.00
73409,Oslo Coffee Roasters,unknown,open,True,0.85,0.85,0.00
67915,Atlantic Diner,unknown,open,True,0.85,0.85,0.00
84432,Oslo Coffee Roasters,unknown,open,True,0.85,0.85,0.00
38520,Indian Head Trailhead,outdoor,unknown,True,0.75,0.30,-0.45


## Verifying Contextual Scores and Activity Identifiers

In [23]:
# Check activity identifier uniqueness
print("TOTAL ACTIVITIES:", len(activity_df))
print("UNIQUE GMAP IDs:", activity_df["gmap_id"].nunique())
print("DUPLICATED GMAP IDs:", activity_df["gmap_id"].duplicated().sum())

# Verify scores remain within [0, 1]
score_columns = [
    "context_clear_comfortable",
    "context_rainy_cold"
]

for column in score_columns:
    print(
        f"{column}:",
        "Min =", activity_df[column].min(),
        "Max =", activity_df[column].max(),
        "Missing =", activity_df[column].isna().sum()
    )

# Inspect the trailhead record
print("\nINDIAN HEAD TRAILHEAD SCORES")

display(
    activity_df.loc[
        activity_df["name"] == "Indian Head Trailhead",
        [
            "gmap_id",
            "name",
            "weather_exposure_v2",
            "time_score",
            "context_clear_comfortable",
            "context_rainy_cold"
        ]
    ]
)

# Verify scores for eligible activities
eligible_df = activity_df[
    activity_df["context_eligible"]
]

print("\nELIGIBLE ACTIVITY SCORE SUMMARY")

display(
    eligible_df[score_columns].describe()
)

TOTAL ACTIVITIES: 86577
UNIQUE GMAP IDs: 86354
DUPLICATED GMAP IDs: 223
context_clear_comfortable: Min = 0.35 Max = 1.0 Missing = 0
context_rainy_cold: Min = 0.05 Max = 1.0 Missing = 0

INDIAN HEAD TRAILHEAD SCORES


,gmap_id,name,weather_exposure_v2,time_score,context_clear_comfortable,context_rainy_cold
38520,0x4ccae9f589e59185:0x90880f88ae33e85a,Indian Head Trailhead,outdoor,0.5,0.75,0.3



ELIGIBLE ACTIVITY SCORE SUMMARY


,context_clear_comfortable,context_rainy_cold
count,75181.000000,75181.000000
mean,0.828558,0.786422
std,0.089695,0.153582
min,0.600000,0.300000
25%,0.850000,0.850000
50%,0.850000,0.850000
75%,0.850000,0.850000
max,1.000000,1.000000


## Investigating Duplicate Activity Identifiers

In [24]:
# Find all records sharing a gmap_id
duplicate_activities = activity_df[
    activity_df["gmap_id"].duplicated(keep=False)
].copy()

print("TOTAL RECORDS WITH DUPLICATED GMAP IDs:")
print(len(duplicate_activities))

print("\nNUMBER OF DISTINCT DUPLICATED GMAP IDs:")
print(duplicate_activities["gmap_id"].nunique())

print("\nDUPLICATE GROUP SIZES:")
display(
    duplicate_activities.groupby("gmap_id")
    .size()
    .value_counts()
    .sort_index()
    .to_frame("Number of IDs")
)

# Check whether duplicate IDs have different names or coordinates
duplicate_summary = (
    duplicate_activities.groupby("gmap_id")
    .agg(
        record_count=("name", "size"),
        unique_names=("name", "nunique"),
        unique_latitudes=("latitude", "nunique"),
        unique_longitudes=("longitude", "nunique"),
        unique_hours=("hours", "nunique")
    )
    .reset_index()
)

print("\nDUPLICATE METADATA CONSISTENCY")
display(
    duplicate_summary[
        [
            "record_count",
            "unique_names",
            "unique_latitudes",
            "unique_longitudes",
            "unique_hours"
        ]
    ].describe()
)

print("\nSAMPLE DUPLICATE RECORDS")
display(
    duplicate_activities[
        [
            "gmap_id",
            "name",
            "latitude",
            "longitude",
            "hours",
            "weather_exposure_v2"
        ]
    ].sort_values("gmap_id").head(12)
)

TOTAL RECORDS WITH DUPLICATED GMAP IDs:
446

NUMBER OF DISTINCT DUPLICATED GMAP IDs:
223

DUPLICATE GROUP SIZES:


,Number of IDs
2,223



DUPLICATE METADATA CONSISTENCY


,record_count,unique_names,unique_latitudes,unique_longitudes,unique_hours
count,223.0,223.0,223.0,223.0,223.00000
mean,2.0,1.0,1.0,1.0,0.77130
std,0.0,0.0,0.0,0.0,0.42094
min,2.0,1.0,1.0,1.0,0.00000
25%,2.0,1.0,1.0,1.0,1.00000
50%,2.0,1.0,1.0,1.0,1.00000
75%,2.0,1.0,1.0,1.0,1.00000
max,2.0,1.0,1.0,1.0,1.00000



SAMPLE DUPLICATE RECORDS


,gmap_id,name,latitude,longitude,hours,weather_exposure_v2
260,0x4ccab4b3af9c7c85:0x9defa19fe4e295c,Alpine Pizza,43.837433,-73.761254,"[['Wednesday', '4–9PM'], ['Thursday', '4–9PM']...",unknown
37,0x4ccab4b3af9c7c85:0x9defa19fe4e295c,Alpine Pizza,43.837433,-73.761254,"[['Wednesday', '4–9PM'], ['Thursday', '4–9PM']...",unknown
293,0x4ccab4b3e72035e7:0x736a2307cb7b17af,Schroon Lake Camps,43.855169,-73.759335,NaN,outdoor
70,0x4ccab4b3e72035e7:0x736a2307cb7b17af,Schroon Lake Camps,43.855169,-73.759335,NaN,outdoor
38,0x4ccab7755367e6b3:0xfb5438273acb64bc,North Hudson Beach Park,43.959307,-73.724186,"[['Wednesday', '6AM–9PM'], ['Thursday', '6AM–9...",outdoor
261,0x4ccab7755367e6b3:0xfb5438273acb64bc,North Hudson Beach Park,43.959307,-73.724186,"[['Wednesday', '6AM–9PM'], ['Thursday', '6AM–9...",outdoor
417,0x4ccaf93f0974c325:0x236fd0ed8e84c49f,Wilmington Recreation Park,44.386957,-73.814043,"[['Wednesday', '7AM–9:30PM'], ['Thursday', '7A...",outdoor
194,0x4ccaf93f0974c325:0x236fd0ed8e84c49f,Wilmington Recreation Park,44.386957,-73.814043,"[['Wednesday', '7AM–9:30PM'], ['Thursday', '7A...",outdoor
418,0x4ccb056a335054c9:0x2a41677a9e9a9002,Mt. Pisgah Recreation Center,44.345443,-74.125165,"[['Wednesday', 'Open 24 hours'], ['Thursday', ...",outdoor
195,0x4ccb056a335054c9:0x2a41677a9e9a9002,Mt. Pisgah Recreation Center,44.345443,-74.125165,"[['Wednesday', 'Open 24 hours'], ['Thursday', ...",outdoor


## Checking Duplicate Record Consistency

In [25]:
# Check only original dataset columns, excluding derived columns
original_columns = [
    "name", "address", "gmap_id", "description",
    "latitude", "longitude", "category", "avg_rating",
    "num_of_reviews", "price", "hours", "MISC",
    "state", "relative_results", "url",
    "is_activity", "activity_groups"
]

# Count exact duplicate rows
exact_duplicates = duplicate_activities[
    original_columns
].duplicated().sum()

print("EXACT DUPLICATE ROWS:", exact_duplicates)

# Check differences within each duplicate ID
metadata_columns = [
    col for col in original_columns
    if col != "gmap_id"
]

conflicting_fields = {}

for column in metadata_columns:
    unique_counts = (
        duplicate_activities.groupby("gmap_id")[column]
        .nunique(dropna=False)
    )

    conflicting_fields[column] = int(
        (unique_counts > 1).sum()
    )

print("\nNUMBER OF DUPLICATE IDs WITH DIFFERENCES BY FIELD:")

display(
    pd.DataFrame(
        conflicting_fields.items(),
        columns=["Field", "IDs With Differences"]
    ).sort_values(
        "IDs With Differences",
        ascending=False
    )
)

print("\nTOTAL DUPLICATE ID GROUPS WITH ANY DIFFERENCES:")

print(
    sum(
        any(
            group[column].nunique(dropna=False) > 1
            for column in metadata_columns
        )
        for _, group in duplicate_activities.groupby("gmap_id")
    )
)

EXACT DUPLICATE ROWS: 223

NUMBER OF DUPLICATE IDs WITH DIFFERENCES BY FIELD:


,Field,IDs With Differences
0,name,0
1,address,0
2,description,0
3,latitude,0
4,longitude,0
5,category,0
6,avg_rating,0
7,num_of_reviews,0
8,price,0
9,hours,0



TOTAL DUPLICATE ID GROUPS WITH ANY DIFFERENCES:
0


## Preparing the Context-Scoring Dataset



In [26]:
# Select columns needed for context integration
context_columns = [
    "gmap_id",
    "name",
    "latitude",
    "longitude",
    "weather_exposure_v2",
    "time_availability",
    "time_score",
    "context_clear_comfortable",
    "context_rainy_cold",
    "context_eligible"
]

# Create a separate context-scoring table
context_df = (
    activity_df[context_columns]
    .drop_duplicates(subset="gmap_id")
    .reset_index(drop=True)
    .copy()
)

print("ORIGINAL DATASET SHAPE:")
print(activity_df.shape)

print("\nCONTEXT DATASET SHAPE:")
print(context_df.shape)

print("\nUNIQUE ACTIVITY IDs:")
print(context_df["gmap_id"].nunique())

print("\nDUPLICATED ACTIVITY IDs:")
print(context_df["gmap_id"].duplicated().sum())

print("\nMISSING VALUES:")
display(context_df.isna().sum().to_frame("Missing Values"))

print("\nSAMPLE CONTEXT RECORDS:")
display(context_df.sample(10, random_state=42))

ORIGINAL DATASET SHAPE:
(86577, 31)

CONTEXT DATASET SHAPE:
(86354, 10)

UNIQUE ACTIVITY IDs:
86354

DUPLICATED ACTIVITY IDs:
0

MISSING VALUES:


,Missing Values
gmap_id,0
name,0
latitude,0
longitude,0
weather_exposure_v2,0
time_availability,0
time_score,0
context_clear_comfortable,0
context_rainy_cold,0
context_eligible,0



SAMPLE CONTEXT RECORDS:


,gmap_id,name,latitude,longitude,weather_exposure_v2,time_availability,time_score,context_clear_comfortable,context_rainy_cold,context_eligible
36804,0x89c258f0a0dc2135:0xcd7579aa07e83618,Sothys Spa,40.764038,-73.976077,indoor,open,1.0,0.95,1.00,True
81551,0x89d190a683c01dad:0xc20199267c8bd708,Applebee's Grill + Bar,42.348298,-77.668242,unknown,open,1.0,0.85,0.85,True
56735,0x89d04271b483c8fd:0x8f6fdbc5fa67f61f,Mark Twain,42.136039,-76.829707,unknown,open,1.0,0.85,0.85,True
67287,0x89d9f38e6be378a5:0x95da939988de43f8,KFC,43.031130,-76.148288,unknown,open,1.0,0.85,0.85,True
60579,0x89e894db3b0de1c3:0x6c46d3f9982aa327,Southampton History Museum,40.884731,-72.388603,indoor,open,1.0,0.95,1.00,True
38442,0x89c260d1bcd9726f:0xa42056ec25e3fdc3,Shake n Burger,40.696417,-73.804503,unknown,open,1.0,0.85,0.85,True
18473,0x89c258bf356f6ff9:0x3f244f67282bd373,Eastside Massage Therapy,40.772098,-73.953646,unknown,open,1.0,0.85,0.85,True
48132,0x89d7432eb17a0f29:0x6181c5cc211d5be6,The Got Away,43.320102,-76.726265,outdoor,unknown,0.5,0.75,0.30,True
14767,0x89dd376199214921:0xd3238ad9432a3a25,Fun Central Inc,41.600218,-73.910487,unknown,unknown,0.5,0.60,0.60,True
72565,0x89daef147ccfe38f:0x2a5f5c32653d0aac,Paul & Sons Pizza,42.095646,-75.926244,unknown,open,1.0,0.85,0.85,True


## Dynamic Weather and Time Context Scoring



In [27]:
def generate_context_scores(
    activities,
    weather_condition,
    temperature_c,
    day,
    hour,
    minute=0
):
    # Work on a separate DataFrame
    result = activities[
        [
            "gmap_id",
            "name",
            "latitude",
            "longitude",
            "weather_exposure_v2",
            "parsed_hours"
        ]
    ].drop_duplicates(subset="gmap_id").copy()

    # Calculate weather suitability
    result["weather_score"] = (
        result["weather_exposure_v2"].apply(
            lambda exposure: calculate_combined_weather_score(
                exposure,
                weather_condition,
                temperature_c
            )
        )
    )

    # Determine operating-hour availability
    result["time_availability"] = (
        result["parsed_hours"].apply(
            lambda hours: check_activity_availability(
                hours, day, hour, minute
            )
        )
    )

    # Convert availability to a numerical score
    result["time_score"] = (
        result["time_availability"].map(time_score_rules)
    )

    # Combine weather and time
    result["context_score"] = (
        0.5 * result["weather_score"]
        + 0.5 * result["time_score"]
    )

    # Exclude confirmed-closed activities
    result["context_eligible"] = (
        result["time_availability"] != "closed"
    )

    return result.reset_index(drop=True)


# Test the function with a different scenario
dynamic_context_df = generate_context_scores(
    activities=activity_df,
    weather_condition="rain",
    temperature_c=12,
    day="Friday",
    hour=19
)

print("DYNAMIC CONTEXT DATASET SHAPE:")
print(dynamic_context_df.shape)

print("\nDUPLICATED ACTIVITY IDs:")
print(dynamic_context_df["gmap_id"].duplicated().sum())

print("\nCONTEXT SCORE SUMMARY:")
display(dynamic_context_df["context_score"].describe())

print("\nTIME AVAILABILITY DISTRIBUTION:")
display(
    dynamic_context_df["time_availability"]
    .value_counts()
    .to_frame("Number of Activities")
)

print("\nSAMPLE DYNAMIC CONTEXT SCORES:")
display(dynamic_context_df.sample(10, random_state=42))

DYNAMIC CONTEXT DATASET SHAPE:
(86354, 11)

DUPLICATED ACTIVITY IDs:
0

CONTEXT SCORE SUMMARY:


count    86354.000000
mean         0.696717
std          0.219802
min          0.100000
25%          0.500000
50%          0.850000
75%          0.850000
max          1.000000
Name: context_score, dtype: float64


TIME AVAILABILITY DISTRIBUTION:


,Number of Activities
time_availability,
open,56102
closed,17765
unknown,12487



SAMPLE DYNAMIC CONTEXT SCORES:


,gmap_id,name,latitude,longitude,weather_exposure_v2,parsed_hours,weather_score,time_availability,time_score,context_score,context_eligible
36804,0x89c258f0a0dc2135:0xcd7579aa07e83618,Sothys Spa,40.764038,-73.976077,indoor,"{'Monday': '10AM–7PM', 'Tuesday': '9AM–8PM', '...",1.0,open,1.0,1.00,True
81551,0x89d190a683c01dad:0xc20199267c8bd708,Applebee's Grill + Bar,42.348298,-77.668242,unknown,"{'Saturday': '11AM–11PM', 'Sunday': '11AM–10PM...",0.7,open,1.0,0.85,True
56735,0x89d04271b483c8fd:0x8f6fdbc5fa67f61f,Mark Twain,42.136039,-76.829707,unknown,"{'Tuesday': '7:30AM–8PM', 'Wednesday': '7:30AM...",0.7,open,1.0,0.85,True
67287,0x89d9f38e6be378a5:0x95da939988de43f8,KFC,43.031130,-76.148288,unknown,"{'Sunday': '10:30AM–10PM', 'Monday': '10:30AM–...",0.7,open,1.0,0.85,True
60579,0x89e894db3b0de1c3:0x6c46d3f9982aa327,Southampton History Museum,40.884731,-72.388603,indoor,"{'Saturday': '11AM–4PM', 'Sunday': 'Closed', '...",1.0,closed,0.0,0.50,False
38442,0x89c260d1bcd9726f:0xa42056ec25e3fdc3,Shake n Burger,40.696417,-73.804503,unknown,"{'Sunday': 'Open 24 hours', 'Monday': 'Open 24...",0.7,open,1.0,0.85,True
18473,0x89c258bf356f6ff9:0x3f244f67282bd373,Eastside Massage Therapy,40.772098,-73.953646,unknown,"{'Sunday': '10AM–3PM', 'Monday': '11:15AM–4:30...",0.7,open,1.0,0.85,True
48132,0x89d7432eb17a0f29:0x6181c5cc211d5be6,The Got Away,43.320102,-76.726265,outdoor,{},0.2,unknown,0.5,0.35,True
14767,0x89dd376199214921:0xd3238ad9432a3a25,Fun Central Inc,41.600218,-73.910487,unknown,{},0.7,unknown,0.5,0.60,True
72565,0x89daef147ccfe38f:0x2a5f5c32653d0aac,Paul & Sons Pizza,42.095646,-75.926244,unknown,"{'Monday': 'Closed', 'Tuesday': 'Closed', 'Wed...",0.7,open,1.0,0.85,True


 ## Testing the OpenWeatherMap API Connection



In [32]:
import requests
from getpass import getpass

# Enter your API key securely
api_key = getpass("Enter your OpenWeatherMap API key: ")

# Sample location: New York City
latitude = 40.7128
longitude = -74.0060

api_url = "https://api.openweathermap.org/data/2.5/weather"

params = {
    "lat": latitude,
    "lon": longitude,
    "appid": api_key,
    "units": "metric"
}

response = requests.get(
    api_url,
    params=params,
    timeout=15
)

print("API RESPONSE STATUS:", response.status_code)

if response.status_code == 200:
    weather_data = response.json()

    print("\nCURRENT WEATHER DATA:")
    print("Location:", weather_data["name"])
    print("Temperature (°C):", weather_data["main"]["temp"])
    print("Weather Condition:", weather_data["weather"][0]["main"])
    print("Description:", weather_data["weather"][0]["description"])
    print("Rainfall (last hour, mm):",
          weather_data.get("rain", {}).get("1h", 0))

else:
    print("API request failed.")
    print("Error code:", response.status_code)
    print("Check that your API key is active and authorized.")

Enter your OpenWeatherMap API key:  ········


API RESPONSE STATUS: 200

CURRENT WEATHER DATA:
Location: New York
Temperature (°C): 13.75
Weather Condition: Clear
Description: clear sky
Rainfall (last hour, mm): 0


## Converting OpenWeatherMap Data into Weather Scoring Inputs

In [33]:
def map_openweather_condition(weather_main):
    condition = weather_main.strip().lower()

    adverse_conditions = {
        "rain",
        "drizzle",
        "thunderstorm",
        "snow"
    }

    if condition in adverse_conditions:
        return "rain"

    return "clear"


# Extract weather information from the API response
api_weather_condition = weather_data["weather"][0]["main"]
api_temperature = weather_data["main"]["temp"]

# Convert to the format used by our scoring function
mapped_weather_condition = map_openweather_condition(
    api_weather_condition
)

print("ORIGINAL API WEATHER:", api_weather_condition)
print("MAPPED WEATHER CONDITION:", mapped_weather_condition)
print("TEMPERATURE (°C):", api_temperature)

# Test the existing weather suitability function
print("\nWEATHER SUITABILITY SCORES")

for exposure in ["indoor", "outdoor", "unknown"]:
    score = calculate_combined_weather_score(
        exposure,
        mapped_weather_condition,
        api_temperature
    )

    print(f"{exposure.capitalize()}: {score}")

ORIGINAL API WEATHER: Clear
MAPPED WEATHER CONDITION: clear
TEMPERATURE (°C): 13.75

WEATHER SUITABILITY SCORES
Indoor: 0.9
Outdoor: 1.0
Unknown: 0.7


## Generating Context Scores Using Live Weather Data

In [34]:
from datetime import datetime
from zoneinfo import ZoneInfo

# Extract current weather from the API response
live_weather = map_openweather_condition(
    weather_data["weather"][0]["main"]
)

live_temperature = weather_data["main"]["temp"]

# Use the API observation timestamp, converted to New York time
observation_time = datetime.fromtimestamp(
    weather_data["dt"],
    tz=ZoneInfo("America/New_York")
)

live_day = observation_time.strftime("%A")
live_hour = observation_time.hour
live_minute = observation_time.minute

print("LIVE WEATHER SCENARIO")
print("Location:", weather_data["name"])
print("Observation time:", observation_time)
print("Weather:", live_weather)
print("Temperature (°C):", live_temperature)

# Generate contextual scores for all unique activities
live_context_df = generate_context_scores(
    activities=activity_df,
    weather_condition=live_weather,
    temperature_c=live_temperature,
    day=live_day,
    hour=live_hour,
    minute=live_minute
)

print("\nLIVE CONTEXT DATASET SHAPE:")
print(live_context_df.shape)

print("\nDUPLICATED ACTIVITY IDs:")
print(live_context_df["gmap_id"].duplicated().sum())

print("\nTIME AVAILABILITY DISTRIBUTION:")
display(
    live_context_df["time_availability"]
    .value_counts()
    .to_frame("Number of Activities")
)

print("\nCONTEXT SCORE SUMMARY:")
display(live_context_df["context_score"].describe())

print("\nSAMPLE LIVE CONTEXT SCORES:")
display(
    live_context_df[
        [
            "name",
            "weather_exposure_v2",
            "weather_score",
            "time_availability",
            "time_score",
            "context_score",
            "context_eligible"
        ]
    ].sample(10, random_state=42)
)

LIVE WEATHER SCENARIO
Location: New York
Observation time: 2026-11-04 02:50:28-05:00
Weather: clear
Temperature (°C): 13.75

LIVE CONTEXT DATASET SHAPE:
(86354, 11)

DUPLICATED ACTIVITY IDs:
0

TIME AVAILABILITY DISTRIBUTION:


,Number of Activities
time_availability,
closed,69200
unknown,12487
open,4667



CONTEXT SCORE SUMMARY:


count    86354.000000
mean         0.431518
std          0.154234
min          0.350000
25%          0.350000
50%          0.350000
75%          0.450000
max          1.000000
Name: context_score, dtype: float64


SAMPLE LIVE CONTEXT SCORES:


,name,weather_exposure_v2,weather_score,time_availability,time_score,context_score,context_eligible
36804,Sothys Spa,indoor,0.9,closed,0.0,0.45,False
81551,Applebee's Grill + Bar,unknown,0.7,closed,0.0,0.35,False
56735,Mark Twain,unknown,0.7,closed,0.0,0.35,False
67287,KFC,unknown,0.7,closed,0.0,0.35,False
60579,Southampton History Museum,indoor,0.9,closed,0.0,0.45,False
38442,Shake n Burger,unknown,0.7,open,1.0,0.85,True
18473,Eastside Massage Therapy,unknown,0.7,closed,0.0,0.35,False
48132,The Got Away,outdoor,1.0,unknown,0.5,0.75,True
14767,Fun Central Inc,unknown,0.7,unknown,0.5,0.60,True
72565,Paul & Sons Pizza,unknown,0.7,closed,0.0,0.35,False


## Validating Live Weather and Time Context Scores



In [35]:
import numpy as np

# 1. Check score range and missing values
score_range_valid = live_context_df["context_score"].between(0, 1).all()

missing_scores = live_context_df[
    ["weather_score", "time_score", "context_score"]
].isna().sum()

# 2. Verify the weighted aggregation
expected_scores = (
    0.5 * live_context_df["weather_score"]
    + 0.5 * live_context_df["time_score"]
)

aggregation_valid = np.allclose(
    live_context_df["context_score"],
    expected_scores
)

# 3. Verify closed activities are ineligible
closed_activities = live_context_df[
    live_context_df["time_availability"] == "closed"
]

closed_filter_valid = (
    ~closed_activities["context_eligible"]
).all()

# 4. Verify eligible activities are not classified as closed
eligible_activities = live_context_df[
    live_context_df["context_eligible"]
]

eligibility_valid = (
    eligible_activities["time_availability"] != "closed"
).all()

print("LIVE CONTEXT VALIDATION RESULTS")
print("Scores within [0, 1]:", score_range_valid)
print("Weighted aggregation correct:", aggregation_valid)
print("Closed activities excluded:", closed_filter_valid)
print("Eligible activities not closed:", eligibility_valid)

print("\nMISSING SCORES:")
display(missing_scores.to_frame("Missing Values"))

print("\nELIGIBLE ACTIVITIES:")
print(len(eligible_activities))

print("\nINELIGIBLE ACTIVITIES:")
print(len(closed_activities))

LIVE CONTEXT VALIDATION RESULTS
Scores within [0, 1]: True
Weighted aggregation correct: True
Closed activities excluded: True
Eligible activities not closed: True

MISSING SCORES:


,Missing Values
weather_score,0
time_score,0
context_score,0



ELIGIBLE ACTIVITIES:
17154

INELIGIBLE ACTIVITIES:
69200


## Exporting Weather and Time Context Scores


In [36]:
from pathlib import Path

# Select fields needed for recommendation integration
export_columns = [
    "gmap_id",
    "weather_exposure_v2",
    "weather_score",
    "time_availability",
    "time_score",
    "context_score",
    "context_eligible"
]

context_export_df = live_context_df[export_columns].copy()

# Create a dedicated output directory
output_dir = Path("../data/context")
output_dir.mkdir(parents=True, exist_ok=True)

# Include the observation timestamp in the filename
timestamp_label = observation_time.strftime("%Y%m%d_%H%M")

output_path = (
    output_dir / f"weather_time_context_{timestamp_label}.csv"
)

# Save the context scores
context_export_df.to_csv(output_path, index=False)

print("EXPORTED FILE:")
print(output_path.resolve())

print("\nEXPORTED DATASET SHAPE:")
print(context_export_df.shape)

print("\nUNIQUE ACTIVITY IDs:")
print(context_export_df["gmap_id"].nunique())

print("\nMISSING VALUES:")
display(context_export_df.isna().sum().to_frame("Missing Values"))

print("\nFIRST FIVE ROWS:")
display(context_export_df.head())

EXPORTED FILE:
/Users/ogechikaesedebe/Downloads/Activity-Recommendation-System/data/context/weather_time_context_20261104_0250.csv

EXPORTED DATASET SHAPE:
(86354, 7)

UNIQUE ACTIVITY IDs:
86354

MISSING VALUES:


,Missing Values
gmap_id,0
weather_exposure_v2,0
weather_score,0
time_availability,0
time_score,0
context_score,0
context_eligible,0



FIRST FIVE ROWS:


,gmap_id,weather_exposure_v2,weather_score,time_availability,time_score,context_score,context_eligible
0,0x89d0be3bcc338707:0x3e3d9bf1247ba191,unknown,0.7,closed,0.0,0.35,False
1,0x89c244688b7609cd:0x205e2d8d173dd35e,indoor,0.9,closed,0.0,0.45,False
2,0x89d13fd0bc4468e1:0x1a81c620f5086b11,outdoor,1.0,unknown,0.5,0.75,True
3,0x89c25923cbb1c20d:0xd9edcb0a146ba104,unknown,0.7,open,1.0,0.85,True
4,0x89c261f60bdf13db:0x38da730e4687a97b,unknown,0.7,closed,0.0,0.35,False


## Verifying the Exported Context Data



In [37]:
# Reload the exported CSV
reloaded_context_df = pd.read_csv(output_path)

print("RELOADED DATASET SHAPE:")
print(reloaded_context_df.shape)

print("\nCOLUMN DATA TYPES:")
print(reloaded_context_df.dtypes)

print("\nDUPLICATED ACTIVITY IDs:")
print(reloaded_context_df["gmap_id"].duplicated().sum())

print("\nMISSING VALUES:")
print(reloaded_context_df.isna().sum().sum())

print("\nDATA MATCHES ORIGINAL EXPORT:")
print(reloaded_context_df.equals(context_export_df))

RELOADED DATASET SHAPE:
(86354, 7)

COLUMN DATA TYPES:
gmap_id                    str
weather_exposure_v2        str
weather_score          float64
time_availability          str
time_score             float64
context_score          float64
context_eligible          bool
dtype: object

DUPLICATED ACTIVITY IDs:
0

MISSING VALUES:
0

DATA MATCHES ORIGINAL EXPORT:
True


## Supporting User-Based Weather Retrieval

In [39]:
def get_weather_for_location(latitude, longitude, api_key):
    """
    Retrieve current weather for user-provided coordinates.

    Coordinates may come from automatic location detection
    or manual location selection.
    """

    response = requests.get(
        "https://api.openweathermap.org/data/2.5/weather",
        params={
            "lat": latitude,
            "lon": longitude,
            "appid": api_key,
            "units": "metric"
        },
        timeout=15
    )

    response.raise_for_status()
    data = response.json()

    return {
        "location": data.get("name"),
        "latitude": latitude,
        "longitude": longitude,
        "weather_condition": map_openweather_condition(
            data["weather"][0]["main"]
        ),
        "temperature_c": data["main"]["temp"],
        "observation_timestamp": data["dt"]
    }


# Test with New York City coordinates
test_weather = get_weather_for_location(
    latitude=40.7128,
    longitude=-74.0060,
    api_key=api_key
)

print("LOCATION-BASED WEATHER TEST")

for key, value in test_weather.items():
    print(f"{key}: {value}")

LOCATION-BASED WEATHER TEST
location: New York
latitude: 40.7128
longitude: -74.006
weather_condition: clear
temperature_c: 13.91
observation_timestamp: 1793779796


## Integrating User Location with Dynamic Context Scoring



In [40]:
def generate_location_context_scores(
    activities,
    latitude,
    longitude,
    api_key
):
    # Retrieve current weather for the user's location
    weather_info = get_weather_for_location(
        latitude=latitude,
        longitude=longitude,
        api_key=api_key
    )

    # Convert API observation time to New York local time
    local_time = datetime.fromtimestamp(
        weather_info["observation_timestamp"],
        tz=ZoneInfo("America/New_York")
    )

    # Generate context scores using existing function
    scores = generate_context_scores(
        activities=activities,
        weather_condition=weather_info["weather_condition"],
        temperature_c=weather_info["temperature_c"],
        day=local_time.strftime("%A"),
        hour=local_time.hour,
        minute=local_time.minute
    )

    return weather_info, scores


# Test using New York City coordinates
location_weather, location_context_df = (
    generate_location_context_scores(
        activities=activity_df,
        latitude=40.7128,
        longitude=-74.0060,
        api_key=api_key
    )
)

print("USER LOCATION WEATHER:")
print(location_weather)

print("\nCONTEXT DATASET SHAPE:")
print(location_context_df.shape)

print("\nDUPLICATED ACTIVITY IDs:")
print(location_context_df["gmap_id"].duplicated().sum())

print("\nTIME AVAILABILITY:")
print(location_context_df["time_availability"].value_counts())

print("\nELIGIBLE ACTIVITIES:")
print(location_context_df["context_eligible"].sum())

print("\nAVERAGE CONTEXT SCORE:")
print(round(location_context_df["context_score"].mean(), 4))

USER LOCATION WEATHER:
{'location': 'New York', 'latitude': 40.7128, 'longitude': -74.006, 'weather_condition': 'clear', 'temperature_c': 13.91, 'observation_timestamp': 1793779796}

CONTEXT DATASET SHAPE:
(86354, 11)

DUPLICATED ACTIVITY IDs:
0

TIME AVAILABILITY:
time_availability
closed     69575
unknown    12487
open        4292
Name: count, dtype: int64

ELIGIBLE ACTIVITIES:
16779

AVERAGE CONTEXT SCORE:
0.4293


## Testing Weather Retrieval Across New York Locations

In [41]:
# Brooklyn test coordinates
brooklyn_lat = 40.6782
brooklyn_lon = -73.9442

# Retrieve weather for Brooklyn
brooklyn_weather = get_weather_for_location(
    latitude=brooklyn_lat,
    longitude=brooklyn_lon,
    api_key=api_key
)

print("BROOKLYN WEATHER TEST")

for key, value in brooklyn_weather.items():
    print(f"{key}: {value}")

print("\nCOORDINATE VALIDATION:")
print(
    "Requested coordinates match:",
    brooklyn_weather["latitude"] == brooklyn_lat
    and brooklyn_weather["longitude"] == brooklyn_lon
)

BROOKLYN WEATHER TEST
location: Brooklyn
latitude: 40.6782
longitude: -73.9442
weather_condition: clear
temperature_c: 13.84
observation_timestamp: 1793779830

COORDINATE VALIDATION:
Requested coordinates match: True


## Converting Manual Location Input into Coordinates

In [42]:
def geocode_location(location_name, api_key):
    """
    Convert a manually entered location into coordinates
    using OpenWeatherMap's Geocoding API.
    """

    response = requests.get(
        "https://api.openweathermap.org/geo/1.0/direct",
        params={
            "q": f"{location_name},US",
            "limit": 5,
            "appid": api_key
        },
        timeout=15
    )

    response.raise_for_status()
    results = response.json()

    if not results:
        raise ValueError(
            f"No location found for: {location_name}"
        )

    # Prefer a New York State result
    ny_results = [
        result for result in results
        if result.get("state") == "New York"
    ]

    if not ny_results:
        raise ValueError(
            "No matching location found in New York State."
        )

    location = ny_results[0]

    return {
        "name": location["name"],
        "state": location.get("state"),
        "country": location.get("country"),
        "latitude": location["lat"],
        "longitude": location["lon"]
    }


# Test manual location entry
manual_location = geocode_location(
    location_name="Brooklyn",
    api_key=api_key
)

print("MANUAL LOCATION TEST")

for key, value in manual_location.items():
    print(f"{key}: {value}")

MANUAL LOCATION TEST
name: Brooklyn
state: New York
country: US
latitude: 40.6526006
longitude: -73.9497211


## Integrating Manual Location Entry with Context Scoring

In [43]:
def generate_context_from_location_name(
    activities,
    location_name,
    api_key
):
    # Convert manually entered location to coordinates
    resolved_location = geocode_location(
        location_name=location_name,
        api_key=api_key
    )

    # Retrieve weather and generate context scores
    weather_info, context_scores = generate_location_context_scores(
        activities=activities,
        latitude=resolved_location["latitude"],
        longitude=resolved_location["longitude"],
        api_key=api_key
    )

    return resolved_location, weather_info, context_scores


# Test with a manually entered location
resolved_location, manual_weather, manual_context_df = (
    generate_context_from_location_name(
        activities=activity_df,
        location_name="Brooklyn",
        api_key=api_key
    )
)

print("RESOLVED USER LOCATION:")
print(resolved_location)

print("\nRETRIEVED WEATHER:")
print(manual_weather)

print("\nCONTEXT DATASET SHAPE:")
print(manual_context_df.shape)

print("\nDUPLICATED ACTIVITY IDs:")
print(manual_context_df["gmap_id"].duplicated().sum())

print("\nELIGIBLE ACTIVITIES:")
print(manual_context_df["context_eligible"].sum())

print("\nAVERAGE CONTEXT SCORE:")
print(round(manual_context_df["context_score"].mean(), 4))

RESOLVED USER LOCATION:
{'name': 'Brooklyn', 'state': 'New York', 'country': 'US', 'latitude': 40.6526006, 'longitude': -73.9497211}

RETRIEVED WEATHER:
{'location': 'Brooklyn', 'latitude': 40.6526006, 'longitude': -73.9497211, 'weather_condition': 'clear', 'temperature_c': 13.87, 'observation_timestamp': 1793780217}

CONTEXT DATASET SHAPE:
(86354, 11)

DUPLICATED ACTIVITY IDs:
0

ELIGIBLE ACTIVITIES:
16779

AVERAGE CONTEXT SCORE:
0.4293
